# 01 · R essentials

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: R* - Instructor: Anderson Santos

## Setup — run this first

This cell installs the R packages the lessons use and makes the example data available. On **Google Colab** it just works: run it (Shift+Enter) and wait until it prints **Ready**. You do not need to understand it.

In [ ]:
# Setup — run this first. Works on Google Colab and on a local Jupyter with R.
# It installs the packages the lessons use and makes the example data available.

pkgs <- c("readr", "dplyr", "tidyr", "tibble", "ggplot2", "forcats", "stringr", "vegan")
to_install <- setdiff(pkgs, rownames(installed.packages()))
if (length(to_install) > 0) {
  # On Colab (Linux) use Posit Public Package Manager binaries: ~1-2 min instead of
  # compiling from source (~15-20 min).
  if (grepl("linux", R.version$os) && file.exists("/etc/os-release")) {
    cn <- grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE)
    cn <- gsub('VERSION_CODENAME=|"', "", cn)
    if (length(cn) == 1 && nzchar(cn)) {
      options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", cn)))
      options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
              paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
    }
  }
  install.packages(to_install)
}

# Fetch the example data if it is not already next to the notebook (the Colab case).
if (!file.exists("../data/sample_metadata.csv") && !file.exists("data/sample_metadata.csv")) {
  system("git clone -q https://github.com/andersonavilasantos/ufz-summerschool-r.git course_repo")
  setwd("course_repo/notebooks")
}

# Load the packages quietly and keep read_csv output tidy (cleaner notebook output).
suppressPackageStartupMessages(invisible(lapply(pkgs, require, character.only = TRUE)))
options(readr.show_col_types = FALSE)

cat("Ready. Data folder:", if (file.exists("../data/sample_metadata.csv")) "../data" else "data", "\n")

## Learning objectives

By the end of this lesson you will be able to:

- Run code in a **notebook cell** and read the result printed below it.
- Create objects with the assignment arrow `<-` and inspect their **type**.
- Work with the workhorse of R — the **vector** — and do maths on whole vectors at once.
- Index and subset vectors, and handle **missing values** (`NA`).
- Write and call a small **function**.
- Chain steps together with the **pipe** (`|>` and `%>%`).

> This is a *refresher*: no prior R needed. Every idea is shown with a runnable
> example you can teach straight from the cell.

---

## 0 · How a notebook runs R

**R** is the language (the engine). This lesson is a **notebook** — a document made
of **cells**, run in your browser on **Google Colab** (or in Jupyter locally).
There are two kinds of cell:

- **Text cells** (like this one): explanations, in formatted Markdown.
- **Code cells** (grey): R code you run. The result appears right below the cell.

To run a code cell, click inside it and press **Shift + Enter** — the code runs
and the next cell is selected. Work top to bottom: cells share one R session, so
an object you create in one cell is available in the next. If things get confused,
use **Runtime ▸ Restart session and run all** (Colab) to start clean.

> **Instructor note.** Have participants run the very first code cell below live.
> Seeing a number appear beneath the cell is the "hello world" moment that makes R
> feel real.

In [ ]:
# Anything after a '#' is a comment: R ignores it. Use comments generously.
# The line below is an expression. Run it and R prints the result.
2 + 2

---

## 1 · Objects and assignment

We store values in **objects** using the assignment arrow `<-`
(read it as "gets"). Typing the object's name prints it.

In [ ]:
# Store the number of samples in our dataset in an object called n_samples.
n_samples <- 1006          # '<-' is R's assignment arrow: put 1006 into n_samples
                           # (in R we use '<-', not '='; think of it as "gets")

n_samples                  # type an object's name on its own line -> R prints it

# Objects behave exactly like the values inside them, so we can compute with them.
n_taxa <- 130              # 130 bacterial genera measured in our study
n_samples * n_taxa         # samples x genera = total cells in the abundance table

R has a few basic **types** of value. The function `class()` tells you which:

In [ ]:
# class() reveals the type of a value; getting types wrong is a common beginner bug.
class(3.14)                # "numeric"  -> numbers (decimals); R's default number type
class(42L)                 # "integer"  -> a whole number; the trailing L forces integer
class("Firmicutes")        # "character"-> text; always wrapped in quotes, e.g. a taxon name
class(TRUE)                # "logical"  -> TRUE / FALSE, the result of a yes/no test

> **Instructor note.** Beginners mix up `=` and `<-`. Both can assign, but the
> community convention is `<-` for assignment and `=` only for function
> arguments. On Colab you can type ` <- ` quickly with **Alt+-** (RStudio users
> know the same shortcut).

### Exercise 1

Create an object `phylum` holding the text `"Bacteroidetes"`, and an object
`abundance` holding the number `1523`. Then print each one's `class()`.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
phylum <- "Bacteroidetes"
abundance <- 1523
class(phylum)      # "character"
class(abundance)   # "numeric"
```
</details>

---

## 2 · Vectors — R's core object

A **vector** is an ordered collection of values *of the same type*. You build one
with `c()` ("combine"). Almost everything in R is a vector.

In [ ]:
# c() = "combine": glue values into a vector. All elements share one type.
# Here: five common human-gut genera as a character (text) vector.
genera <- c("Bacteroides", "Prevotella", "Faecalibacterium",
            "Escherichia", "Akkermansia")
genera                    # print it: R shows the 5 elements in order

# Their (made-up) abundances in one sample — a numeric vector, aligned to 'genera'.
counts <- c(1820, 640, 2310, 95, 210)   # element i is the count of genus i
counts

A useful feature of R: operations apply to every element at once. This is called
*vectorisation*, and it means you rarely need a loop.

In [ ]:
# Vectorisation: the operation is applied to every element at once, no loop needed.
counts * 2            # multiply each count by 2 -> a new 5-element vector
counts / sum(counts)  # each count / the total = its proportion of the community
log(counts)           # natural log of each value (common for skewed abundance data)

Handy summary functions work on the whole vector:

In [ ]:
# These reductions take the whole vector and return a single summary number.
length(counts)   # how many elements (here: how many genera)?
sum(counts)      # total read count across all genera
mean(counts)     # average count per genus
max(counts)      # the largest count (the most abundant genus)
min(counts)      # the smallest count (the rarest genus)

### 2.1 · Indexing (picking elements)

R counts **from 1** (not 0). Use square brackets `[ ]` to pick elements.

In [ ]:
# Square brackets pick elements by position. R starts counting at 1, not 0.
genera[1]            # the 1st genus ("Bacteroides")
genera[c(1, 3)]      # pass a vector of positions -> the 1st and 3rd genera
counts[counts > 500] # logical filter: keep only the counts greater than 500

That last line does a lot of work: `counts > 500` produces `TRUE/FALSE`, and R
keeps the `TRUE` positions. You will use this idea constantly.

In [ ]:
# The comparison itself produces a TRUE/FALSE vector, one entry per element:
counts > 500                 # TRUE where the count exceeds 500, FALSE otherwise
# Feed that mask to genera[] to keep the names at the TRUE positions:
genera[counts > 500]         # which genera are abundant (>500) in this sample?

### 2.2 · Missing values (`NA`)

Real data has gaps. R marks a missing value as `NA`. Many functions return `NA`
if any input is `NA` — unless you tell them to ignore it with `na.rm = TRUE`.

In [ ]:
ages <- c(34, 51, NA, 22, 47)   # NA marks the one participant whose age is unknown
mean(ages)                      # returns NA — one missing value "poisons" the mean
mean(ages, na.rm = TRUE)        # na.rm = TRUE drops the NA, then averages the rest
is.na(ages)                     # TRUE exactly at the missing position, FALSE elsewhere

> **Instructor note.** `NA` is not zero and not the text "NA". It is a
> first-class "unknown". This distinction saves people from silent bugs later.

### Exercise 2

From `counts`, (a) compute the proportion each genus makes of the total, and
(b) find the name of the genus with the **highest** count.
*Hint:* `which.max()` gives the position of the maximum.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
counts / sum(counts)          # (a) proportions
genera[which.max(counts)]     # (b) "Faecalibacterium" — the top genus
```
</details>

---

## 3 · Factors — categories with a fixed set of levels

When a character variable represents **categories** (e.g. BMI group), R has a
special type called a **factor**. Factors remember the full set of categories
("levels") and control their order — which matters for tables and plots.

In [ ]:
# Start with plain text categories (one BMI class per participant):
bmi <- c("lean", "obese", "lean", "overweight", "obese", "lean")
# factor() converts text -> a factor; 'levels' fixes the categories and their order
# (we choose lean < overweight < obese, a meaningful order, not alphabetical).
bmi_f <- factor(bmi, levels = c("lean", "overweight", "obese"))
bmi_f
levels(bmi_f)     # inspect the categories, printed in the order we set
table(bmi_f)      # tally how many participants fall in each category

We will lean on factors heavily when we colour and facet plots later.

---

## 4 · Functions — packaging a recipe

A **function** takes inputs (*arguments*), does something, and returns a result.
You have used built-in ones (`mean`, `sum`). You can write your own with
`function(...) { ... }`. The last line is what it returns.

In [ ]:
# Define a function with function(args){ body }. 'x' is the input placeholder.
# This one turns a vector of counts into percentages of the total.
to_percent <- function(x) {
  x / sum(x) * 100      # last expression = the value the function returns
}

to_percent(counts)      # call it like any built-in; 'counts' becomes x inside

Arguments can have **defaults**:

In [ ]:
# Shannon diversity — a real ecology metric we reuse all week.
# 'base = exp(1)' is a default: if the caller omits base, natural log is used.
shannon <- function(counts, base = exp(1)) {
  p <- counts[counts > 0] / sum(counts)  # keep present taxa, convert to proportions
  -sum(p * log(p, base = base))          # the Shannon formula: -sum(p * log p)
}

shannon(counts)            # no 'base' given -> uses the default (natural log, "nats")
shannon(counts, base = 2)  # override the default -> measures diversity in "bits"

### Exercise 3

Write a function `richness(x)` that returns how many taxa are **present**
(count greater than 0) in a sample. Test it on `c(0, 5, 0, 12, 3)`.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
richness <- function(x) {
  sum(x > 0)          # TRUE counts as 1, so this counts the non-zero taxa
}
richness(c(0, 5, 0, 12, 3))   # 3 taxa present
```
</details>

---

## 5 · The pipe — reading code like a sentence

R lets you chain steps with a **pipe**, so `x |> f() |> g()` means "take `x`,
send it to `f`, then send *that* to `g`". It reads left-to-right, like a recipe.

R has two pipes that behave the same for our purposes:

- `|>` — the **native** pipe, built into R (≥ 4.1). Nothing to load.
- `%>%` — the **magrittr** pipe from the tidyverse. You will see it everywhere.

In [ ]:
# Without a pipe: calls nest inside each other, so you read inside-out (awkward).
round(mean(counts), 1)

# With the native pipe: read left-to-right. Read '|>' as the word "then".
counts |> mean() |> round(1)   # take counts, then mean(), then round to 1 decimal

The tidyverse's `%>%` pipe (from **magrittr**) does the same thing. You will see
it in countless tutorials and older code, so it helps to recognise it — the two
are interchangeable for everything we do this session.

In [ ]:
library(dplyr)                 # the %>% pipe comes with the tidyverse (loaded via dplyr)
counts %>% mean() %>% round(1) # identical result to the |> version above

We use the pipe constantly in the wrangling and plotting lessons, because it
lets us write "filter, then group, then summarise" as one readable flow.

> **Instructor note.** The two pipes are interchangeable for everything we do.
> Prefer the native `|>` in new code; recognise `%>%` because it fills older
> tutorials and Stack Overflow answers.

### Exercise 4

Using the pipe, take `counts`, convert it to proportions with your `to_percent()`
function, and round to 1 decimal — all in one chain.

<details><summary><b>Solution</b> (click to expand)</summary>


```r
counts |> to_percent() |> round(1)
```
</details>

---

## Recap

- Store values with `<-`; check their kind with `class()`.
- The **vector** is R's core object; operations are **vectorised** (whole-vector).
- Index with `[ ]` (counting from **1**) and filter with logical conditions.
- Missing data is `NA`; use `na.rm = TRUE` to skip it.
- **Factors** hold ordered categories — key for tables and plots.
- Write reusable **functions**; give arguments **defaults**.
- The **pipe** (`|>` / `%>%`) chains steps into readable, left-to-right flows.

**Next:** in lesson **02** we move from single vectors to whole tables —
importing our real microbiome data as a **tibble** and inspecting it.